# Notebook 02 — Visualisations
**Contributor:** SKYLearn-Innovation Learner 2  
**Mentor:** Dingaan Mahlatse Machethe  

We produce 6 charts that tell the story of SA matric STEM performance:
1. National pass rate trend (line chart)
2. STEM vs Non-STEM comparison (grouped bar)
3. Province ranking (horizontal bar)
4. Subject ranking (horizontal bar)
5. Pass rate heatmap (province × year)
6. Distinctions by subject (bar)

In [ ]:
import sys; sys.path.insert(0, '../src')
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns

from preprocess import load_clean
from analyse import national_trend, province_summary, stem_vs_nonstem, subject_ranking

df = load_clean()
BLUE  = '#1F4E79'
GREEN = '#117A65'
RED   = '#E74C3C'
sns.set_theme(style='whitegrid', palette='muted')
print('Data loaded:', df.shape)

In [ ]:
# Chart 1 — National pass rate trend
trend = national_trend(df)

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(trend['year'], trend['national_pass_rate'], marker='o', color=BLUE, linewidth=2.5)
ax.fill_between(trend['year'], trend['national_pass_rate'], alpha=0.1, color=BLUE)
ax.set_title('South Africa — National Matric Pass Rate (2014–2023)', fontsize=14, fontweight='bold')
ax.set_xlabel('Year')
ax.set_ylabel('Pass Rate (%)')
ax.yaxis.set_major_formatter(mtick.PercentFormatter())
ax.set_ylim(40, 100)
for _, row in trend.iterrows():
    ax.annotate(f"{row['national_pass_rate']:.1f}%", (row['year'], row['national_pass_rate']),
                textcoords='offset points', xytext=(0, 8), ha='center', fontsize=8)
plt.tight_layout()
plt.savefig('../data/processed/chart1_national_trend.png', dpi=150)
plt.show()

In [ ]:
# Chart 2 — STEM vs Non-STEM
sv = stem_vs_nonstem(df)
pivot = sv.pivot(index='year', columns='subject_type', values='avg_pass_rate')

fig, ax = plt.subplots(figsize=(10, 5))
pivot.plot(kind='bar', ax=ax, color=[BLUE, GREEN], width=0.6)
ax.set_title('STEM vs Non-STEM Average Pass Rate by Year', fontsize=14, fontweight='bold')
ax.set_xlabel('Year')
ax.set_ylabel('Avg Pass Rate (%)')
ax.yaxis.set_major_formatter(mtick.PercentFormatter())
ax.set_xticklabels(ax.get_xticklabels(), rotation=0)
ax.legend(title='Category')
plt.tight_layout()
plt.savefig('../data/processed/chart2_stem_vs_nonstem.png', dpi=150)
plt.show()

In [ ]:
# Chart 3 — Province ranking (latest year)
latest = df['year'].max()
prov = province_summary(df, year=latest).sort_values('avg_pass_rate')

fig, ax = plt.subplots(figsize=(10, 6))
colors = [RED if r < 60 else GREEN if r >= 75 else BLUE for r in prov['avg_pass_rate']]
ax.barh(prov['province'], prov['avg_pass_rate'], color=colors)
ax.set_title(f'Province Average Pass Rate — {latest}', fontsize=14, fontweight='bold')
ax.set_xlabel('Avg Pass Rate (%)')
ax.axvline(x=60, color=RED, linestyle='--', label='60% benchmark')
ax.legend()
plt.tight_layout()
plt.savefig('../data/processed/chart3_provinces.png', dpi=150)
plt.show()

In [ ]:
# Chart 4 — Subject ranking
ranking = subject_ranking(df, year=latest).sort_values('avg_pass_rate')

fig, ax = plt.subplots(figsize=(10, 7))
bar_colors = [BLUE if t == 'STEM' else GREEN for t in ranking['subject_type']]
ax.barh(ranking['subject'], ranking['avg_pass_rate'], color=bar_colors)
from matplotlib.patches import Patch
ax.legend(handles=[Patch(color=BLUE, label='STEM'), Patch(color=GREEN, label='Non-STEM')])
ax.set_title(f'Subject Pass Rate Ranking — {latest}', fontsize=14, fontweight='bold')
ax.set_xlabel('Avg Pass Rate (%)')
ax.axvline(x=60, color=RED, linestyle='--', label='60% threshold')
plt.tight_layout()
plt.savefig('../data/processed/chart4_subjects.png', dpi=150)
plt.show()

In [ ]:
# Chart 5 — Heatmap: province × year (Mathematics only)
math_df = df[df['subject'] == 'Mathematics']
heat = math_df.pivot_table(index='province', columns='year', values='pass_rate')

fig, ax = plt.subplots(figsize=(12, 6))
sns.heatmap(heat, annot=True, fmt='.1f', cmap='RdYlGn', vmin=40, vmax=90,
            linewidths=0.5, ax=ax, cbar_kws={'label': 'Pass Rate (%)'})
ax.set_title('Mathematics Pass Rate Heatmap — Province × Year', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../data/processed/chart5_heatmap.png', dpi=150)
plt.show()

In [ ]:
# Chart 6 — Distinctions by subject (latest year)
dist = df[df['year'] == latest].groupby('subject')['distinctions'].sum().sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(10, 5))
dist.plot(kind='bar', ax=ax, color=GREEN)
ax.set_title(f'Total Distinctions by Subject — {latest}', fontsize=14, fontweight='bold')
ax.set_xlabel('Subject')
ax.set_ylabel('Distinctions')
ax.set_xticklabels(ax.get_xticklabels(), rotation=35, ha='right')
plt.tight_layout()
plt.savefig('../data/processed/chart6_distinctions.png', dpi=150)
plt.show()

print('\nAll 6 charts saved to data/processed/')